# M1-02 — CNN 2 blocks

Code tương ứng với `m1_02.py`. Chạy các cell từ trên xuống: import, cấu hình, xử lý ảnh, load dữ liệu, tạo model, train/validation và test.

Train tối đa 50 epoch, dừng khi validation accuracy không tăng trong 15 epoch liên tiếp. Test dùng trọng số có validation accuracy cao nhất giữ trong RAM. Notebook chưa được chạy.


In [ ]:
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

from metrics import calculate_metrics

In [ ]:
# 1. Cấu hình
DATA_DIR = "../../data/VN_trash_classification_preprocessing"
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# 2. Xử lý ảnh
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])


In [ ]:
# 3. Load dữ liệu
train_dataset = ImageFolder(DATA_DIR + "/train", transform=train_transform)
val_dataset = ImageFolder(DATA_DIR + "/val", transform=test_transform)
test_dataset = ImageFolder(DATA_DIR + "/test", transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {"num_workers": 8, "pin_memory": True, "persistent_workers": True}
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, **loader_options)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))




In [ ]:
# 4. Tạo CNN 2 blocks
class CNN(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        self.layers = nn.Sequential(
            # [B, 3, 224, 224] -> [B, 32, 112, 112]
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 32, 112, 112] -> [B, 64, 56, 56]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # [B, 64, 56, 56] -> [B, 64, 4, 4] -> [B, 1024]
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, number_of_classes)
        )

    def forward(self, images):
        return self.layers(images)


model = CNN(len(train_dataset.classes)).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)




In [ ]:
# 5. Train và Validation
best_val_accuracy = -1.0
best_weights = None
best_epoch = 0
patience = 15
epochs_without_improvement = 0

for epoch in range(50):
    model.train()
    train_loss = 0
    train_true_labels = []
    train_predicted_labels = []

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)

        train_true_labels.extend(labels.cpu().tolist())
        train_predicted_labels.extend(predictions.cpu().tolist())

    train_loss /= len(train_dataset)
    train_metrics = calculate_metrics(train_true_labels, train_predicted_labels)
    train_accuracy = train_metrics["accuracy"]

    model.eval()
    val_loss = 0
    val_true_labels = []
    val_predicted_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            val_true_labels.extend(labels.cpu().tolist())
            val_predicted_labels.extend(predictions.cpu().tolist())

    val_loss /= len(val_dataset)
    val_metrics = calculate_metrics(val_true_labels, val_predicted_labels)
    val_accuracy = val_metrics["accuracy"]

    # Sao chép trọng số sang RAM để các epoch sau không ghi đè lên bản tốt nhất.
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_weights = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        best_epoch = epoch + 1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1}/50 | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}"
    )

    # Dừng khi validation accuracy không tăng trong 15 epoch liên tiếp.
    if epochs_without_improvement >= patience:
        print(f"Early stopping tại epoch {epoch + 1}")
        break

# Vòng lặp kết thúc khi model đang giữ trọng số của epoch cuối, nạp lại trọng số tốt nhất.
model.load_state_dict(best_weights)




In [ ]:
# 6. Test
print(f"Best epoch: {best_epoch} | Best Val Acc: {best_val_accuracy:.2%}")
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print("\nTest Result")
print(f"Test Accuracy: {metrics['accuracy']:.2%}")
print(f"Macro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall: {metrics['macro_recall']:.4f}")
print(f"Macro F1: {metrics['macro_f1']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
print()
print(classification_report(true_labels, predicted_labels, target_names=test_dataset.classes, digits=4, zero_division=0))

In [ ]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR + "/m1_02.pt")